# Pre-label `dataset_enemy_icons` (`img####.png` only)
Loads the latest model checkpoint and prints top-1 prediction + probability.
This notebook does not rename or modify files.

In [4]:
from pathlib import Path
import re
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

In [5]:
PROJECT_ROOT = Path("../")
CKPT_DIR = PROJECT_ROOT / "checkpoints"
DATASET_DIR = PROJECT_ROOT / "dataset_enemy_icons"

IMG_RE = re.compile(r"^img\d{4}\.png$", re.IGNORECASE)

ckpts = sorted(CKPT_DIR.glob("unit_resnet18_*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
if not ckpts:
    ckpts = sorted(CKPT_DIR.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
ckpt_path = ckpts[0]
print("Using checkpoint:", ckpt_path)

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
assert img_paths, f"No img####.png files found in {DATASET_DIR}"
print(f"Found {len(img_paths)} target files")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
payload = torch.load(ckpt_path, map_location=device)
class_names = payload["class_names"]
image_size = int(payload.get("image_size", 128))
mean = np.array(payload.get("normalize_mean", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.array(payload.get("normalize_std", [0.5, 0.5, 0.5]), dtype=np.float32)
std = np.where(std == 0, 1.0, std)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(payload["model_state_dict"], strict=True)
model = model.to(device)
model.eval()

def preprocess_icon_bgr(img):
    resized = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - mean) / std
    chw = np.transpose(rgb, (2, 0, 1))
    return torch.from_numpy(chw).unsqueeze(0).float().to(device)

for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"{p.name} | READ_FAIL | prob=0.0000")
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    print(f"{p.name} | {class_names[int(i)]} | prob={float(v):.4f}")


Using checkpoint: ..\checkpoints\unit_resnet18_20260317_165735.pt
Found 70 target files
img0001.png | Cerberus | prob=0.9535
img0002.png | Hades | prob=0.2953
img0003.png | Forbidden_Valnesia | prob=0.8734
img0004.png | Winged_Slime | prob=0.9156
img0005.png | Goblin_Crew | prob=0.4386
img0006.png | Ice_Queen_Anatoyu | prob=0.9724
img0007.png | Cerberus | prob=0.9507
img0008.png | Mad_Frog | prob=0.9097
img0009.png | Lamp_Spirit | prob=0.5987
img0010.png | Grilled_Bird | prob=0.7579
img0011.png | Cerberus | prob=0.8840
img0012.png | Hades | prob=0.2650
img0013.png | Forbidden_Valnesia | prob=0.8946
img0014.png | Winged_Slime | prob=0.9119
img0015.png | Goblin_Crew | prob=0.4887
img0016.png | Ice_Queen_Anatoyu | prob=0.9800
img0017.png | Cerberus | prob=0.9068
img0018.png | Mad_Frog | prob=0.9232
img0019.png | Lamp_Spirit | prob=0.4663
img0020.png | Grilled_Bird | prob=0.7976
img0021.png | Cerberus | prob=0.9570
img0022.png | Hades | prob=0.2502
img0023.png | Forbidden_Valnesia | prob=0

In [6]:
ACCEPT_THRESHOLD = 0.40

def next_labeled_name(folder: Path, class_name: str):
    pat = re.compile(rf"^{re.escape(class_name)}_(\\d+)\\.png$", re.IGNORECASE)
    max_idx = 0
    for p in folder.iterdir():
        if not p.is_file():
            continue
        m = pat.match(p.name)
        if m:
            max_idx = max(max_idx, int(m.group(1)))

    idx = max_idx + 1
    while True:
        cand = f"{class_name}_{idx:04d}.png"
        if not (folder / cand).exists():
            return cand
        idx += 1

renamed = 0
kept = 0

img_paths = sorted([p for p in DATASET_DIR.glob("*.png") if IMG_RE.match(p.name)])
for p in img_paths:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        print(f"KEEP {p.name} | READ_FAIL")
        kept += 1
        continue

    inp = preprocess_icon_bgr(img)
    with torch.no_grad():
        logits = model(inp)
        probs = torch.softmax(logits, dim=1)[0]

    v, i = torch.max(probs, dim=0)
    prob = float(v)
    pred_name = class_names[int(i)]

    if prob >= ACCEPT_THRESHOLD:
        new_name = next_labeled_name(DATASET_DIR, pred_name)
        new_path = DATASET_DIR / new_name
        p.rename(new_path)
        print(f"RENAME {p.name} -> {new_name} | prob={prob:.4f}")
        renamed += 1
    else:
        print(f"KEEP {p.name} | pred={pred_name} | prob={prob:.4f}")
        kept += 1

print(f"done | renamed={renamed} kept={kept}")


RENAME img0001.png -> Cerberus_0010.png | prob=0.9535
KEEP img0002.png | pred=Hades | prob=0.2953
RENAME img0003.png -> Forbidden_Valnesia_0012.png | prob=0.8734
RENAME img0004.png -> Winged_Slime_0016.png | prob=0.9156
RENAME img0005.png -> Goblin_Crew_0011.png | prob=0.4386
RENAME img0006.png -> Ice_Queen_Anatoyu_0011.png | prob=0.9724
RENAME img0007.png -> Cerberus_0011.png | prob=0.9507
RENAME img0008.png -> Mad_Frog_0017.png | prob=0.9097
RENAME img0009.png -> Lamp_Spirit_0015.png | prob=0.5987
RENAME img0010.png -> Grilled_Bird_0019.png | prob=0.7579
RENAME img0011.png -> Cerberus_0012.png | prob=0.8840
KEEP img0012.png | pred=Hades | prob=0.2650
RENAME img0013.png -> Forbidden_Valnesia_0013.png | prob=0.8946
RENAME img0014.png -> Winged_Slime_0017.png | prob=0.9119
RENAME img0015.png -> Goblin_Crew_0012.png | prob=0.4887
RENAME img0016.png -> Ice_Queen_Anatoyu_0012.png | prob=0.9800
RENAME img0017.png -> Cerberus_0013.png | prob=0.9068
RENAME img0018.png -> Mad_Frog_0018.png | p